# Analiza sentymentu

Bardzo często spotykanym zagadnieniem w obszarze NLP jest analiza sentymentu danej treści. Jest to zadanie klasyfikacyjne, którego celem jest ocena, czy dany tekst ma pozytywny, czy negatywny wydźwięk. Jest to szczególnie przydatne przy analizie krótkich form pisanych, takich jak komentarze czy opinie internetowe. Dzięki wynikowi takiej analizy właściciel danego portalu może ocenić bezpośrednio, czy to, co na nim się znajduje, podoba się jego użytkownikom.

W kolejnym zadaniu rozwiążemy ten problem z użyciem biblioteki Keras (sugerowana wersja, to 2.3.1), a danymi, których użyjemy, będą komentarze użytkowników dotyczące filmów z anglojęzycznego portalu IMDB. To zadanie będzie o tyle przyjemne, że ten zbiór danych dostarczony jest wraz z biblioteką Keras i przygotowany do trenowania modeli uczenia maszynowego. Model, którego używamy dzisiaj, jest siecią neuronową, o której dowiesz się więcej w kolejnych rozdziałach. Sposób postępowania jednak w tym przypadku jest bardzo podobny do tego ze znanej Ci już biblioteki sklearn. Najpierw będziemy tworzyć obiekt modelu, a następnie trenować go za pomocą metody fit.

Ciekawą częścią tego modułu będzie też zrozumienie, w jaki sposób transformujemy tekst tak, aby był zrozumiały dla sieci neuronowej. Rozpocznijmy zatem od zaimportowania odpowiednich modułów TensorFlow oraz Keras.

In [6]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense
from tensorflow.keras.regularizers import l2

# przygotowanie dannych

Aby przekształcić tekst na postać zrozumiałą dla sieci neuronowej, musimy go najpierw zakodować w postaci liczb. Robimy to w bardzo prosty sposób, to znaczy dla każdego oddzielnego słowa wybieramy jedną liczbę naturalną, która w sposób unikalny je identyfikuje, na przykład "the" -> 1 albo "dog" -> 25. Twórcy biblioteki Keras przygotowali dla nas taki słownik. Możemy go wyświetlić, używając polecenia imdb.get_word_index(). Dodatkowo jest on przygotowany w taki sposób, że słowa w nim posortowane są według kolejności występowania, tzn. słowo przyporządkowane do wartości 1 jest najczęściej występującym słowem (w tym przypadku "the"), słowo przyporządkowane do 2 drugim najczęściej występującym (tutaj "a"). Jest to dla nas o tyle ważne, że sieć neuronowa jest w stanie nauczyć się tylko tych słów, które często występują. Rzadko pojawiające się będą dla niej tylko szumem i nie będą wnosiły dużej wartości. Jest to jeden z tzw. hiperparametrów modelu. Możemy sterować jakością wytrenowanego modelu poprzez wybór, jak dużo słów będzie on rozpoznawał. W naszym przypadku użyjmy 5000 najczęściej występujących wyrazów.

Zrobimy to przez wbudowaną w Kerasa funkcję imdb.load_data(num_words=num_words). Zobaczmy, co ona zwraca.

In [9]:
from tensorflow.keras.datasets import imdb  # import zbioru recenzji filmowych IMDB

# Definicja hiperparametrów
num_words = 5000  # liczba najczęściej występujących słów używanych w słowniku
maxlen = 200  # maksymalna długość pojedynczej recenzji
embedding_dim = 16  # liczba wymiarów wektora embeddingu

# Pobranie danych
(x_train, y_train), (x_test, y_test) = imdb.load_data(
    num_words=num_words
)

# Wyświetlenie pierwszej recenzji zapisanej jako lista indeksów słów
print(x_train[0])

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 2, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 2, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 2, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 2, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 2, 19, 178, 32]


Funkcja imdb.load_data zwraca zbiory treningowe, testowe oraz klasyfikacje danych tekstów jako negatywne lub pozytywne. Jest to postać przetransformowana już tak, aby sieć neuronowa mogła jej użyć. Abyśmy my zobaczyli, jak wygląda faktyczny komentarz użytkownika portalu IMDB, musimy zamienić je z powrotem na tekst.

Wektor ten ma pewne dodatkowe metadane oprócz tekstu. Wartość 1 jest znacznikiem początku tekstu, a wartość 2 oznacza słowo spoza słownika. W tym przypadku wartość 4 oznacza najczęściej występujące słowo (wyraz "the"). Z takiego powodu w funkcji do transformacji jest zagadkowe odejmowanie wartości 3 od indeksu. Więcej informacji tutaj: https://keras.io/api/datasets/imdb/. Aby zobaczyć, jak wygląda komentarz, użyjmy poniższej funkcji. Wartość "#" oznacza słowo spoza słownika, czyli występujące poza listą 5000 najczęściej występujących słów.

In [13]:
# Funkcja zamieniająca wektor indeksów słów na czytelną recenzję tekstową
def vector_to_text(imdb_vector, label):

    reverse_index = {v: k for k, v in imdb.get_word_index().items()}  # utworzenie słownika odwrotnego: indeks -> słowo

    return f"""Comment: {" ".join([reverse_index.get(i - 3, "#") for i in imdb_vector])}
            Label: {"Positive" if label == 1 else "Negative"}"""  # zamiana indeksów na słowa oraz dodanie etykiety recenzji


# Wyświetlenie trzech pierwszych recenzji wraz z ich etykietami
vector_to_text(x_train[0], y_train[0]), vector_to_text(x_train[1], y_train[1]), vector_to_text(x_train[2], y_train[2])  # prezentacja przykładowych danych

("Comment: # this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert # is an amazing actor and now the same being director # father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for # and would recommend it to everyone to watch and the fly # was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also # to the two little # that played the # of norman and paul they were just brilliant children are often left out of the # list i think because the stars that play them all grown up are such a big # for the whole film but these children are amazing and should be # for what they have done don't you think the whol

Funkcja vector_to_text() zamienia recenzję zapisaną jako listę indeksów słów z bazy IMDB na czytelny tekst. Dodatkowo wyświetla etykietę określającą, czy recenzja jest pozytywna (Positive) czy negatywna (Negative).

Pierwsze dane z naszego zbioru treningowego wyglądają zatem następująco:

Co robi i - 3?

Warto o tym pamiętać, bo często pojawia się na zajęciach.

TensorFlow w zbiorze IMDB rezerwuje pierwsze trzy indeksy:

0 – <PAD> (dopełnienie sekwencji),
1 – <START> (początek recenzji),
2 – <UNK> (nieznane słowo).

Dlatego podczas zamiany indeksów na słowa należy odjąć 3, aby dopasować je do właściwego słownika (reverse_index). Dzięki temu zamiast samych liczb otrzymujemy oryginalny tekst recenzji.

In [15]:
# Ujednolicenie długości wszystkich recenzji do maksymalnie 200 słów

x_train = pad_sequences(x_train, maxlen=maxlen)  # uzupełnienie lub skrócenie recenzji treningowych do długości maxlen

x_test = pad_sequences(x_test, maxlen=maxlen)  # uzupełnienie lub skrócenie recenzji testowych do długości maxlen

Funkcja pad_sequences() ujednolica długość wszystkich recenzji. Krótsze recenzje są uzupełniane zerami (padding), natomiast dłuższe są skracane do zadanej długości. Dzięki temu wszystkie próbki mają jednakowy rozmiar i mogą zostać przekazane do sieci neuronowej.

In [16]:
print(x_train.shape)  # wyświetlenie wymiarów zbioru treningowego po zastosowaniu paddingu

print(x_test.shape)  # wyświetlenie wymiarów zbioru testowego po zastosowaniu paddingu

(25000, 200)
(25000, 200)


# trenowanie modelu

Teraz możemy już stworzyć model sieci neuronowej. Nie przejmuj się, jeśli nie rozumiesz dokładnie poniższego kodu. Zrozumiesz to po kolejnym module. Najważniejsze jest, że zwraca on obiekt model, na którym możesz wykonać metodę fit.

In [17]:
# Funkcja tworząca model sieci neuronowej do klasyfikacji recenzji
def build_keras_model(input_dim, output_dim):

    model = Sequential()  # utworzenie pustego modelu sekwencyjnego

    # Warstwa embedding zamienia indeksy słów na wektory liczbowe
    model.add(
        Embedding(
            input_dim=input_dim,  # liczba słów w słowniku
            output_dim=output_dim,  # rozmiar wektora embeddingu
            embeddings_regularizer=l2(0.01)  # regularyzacja L2 zapobiegająca przeuczeniu
        )
    )

    model.add(Flatten())  # spłaszczenie macierzy embeddingów do jednego wektora

    model.add(
        Dense(
            1,  # jeden neuron wyjściowy
            activation='sigmoid'  # funkcja aktywacji dla klasyfikacji binarnej
        )
    )

    model.compile(
        optimizer='adam',  # optymalizator Adam
        loss='binary_crossentropy',  # funkcja straty dla klasyfikacji binarnej
        metrics=['accuracy']  # metryka oceny modelu
    )

    return model  # zwrócenie gotowego modelu


# Utworzenie modelu z wcześniej zdefiniowanymi hiperparametrami
model = build_keras_model(num_words, embedding_dim)

Tworzona jest prosta sieć neuronowa do klasyfikacji recenzji filmowych. Model składa się z warstwy Embedding, która zamienia indeksy słów na wektory liczbowe, warstwy Flatten, która przekształca dane do postaci jednowymiarowej, oraz warstwy Dense z funkcją aktywacji sigmoid, odpowiedzialnej za klasyfikację binarną (pozytywna lub negatywna recenzja). Dodatkowo zastosowano regularyzację L2, aby ograniczyć przeuczenie modelu.

Krótkie wyjaśnienie warstw
- Embedding – zamienia numery słów na gęste wektory liczbowe, które opisują znaczenie słów.
- Flatten – zamienia macierz embeddingów na jeden długi wektor.
- Dense(1, sigmoid) – oblicza prawdopodobieństwo, że recenzja jest pozytywna (1) lub negatywna (0).
- Adam – optymalizator aktualizujący wagi modelu podczas uczenia.
- binary_crossentropy – funkcja straty stosowana w problemach klasyfikacji binarnej.

Teraz możemy już wytrenować nasz model! Do sprawdzenia, czy proces nauczania odbywa się poprawnie, użyjemy zbioru testowego x_test oraz y_test. W kolejnych epokach trafność rozpoznawania naszego modelu w postaci metryki accuracy powinna się zwiększać i ostatecznie osiągnąć wartość około 85%.

Możemy teraz sprawdzić, ile faktycznie parametrów ma nasza sieć. Polecenie model.summary() poinformuje nas, że będzie to 83201 wartości do wytrenowania. Jest to prawie milion razy mniej niż największe obecnie dostępne otwarte modele, takie jak Llama3 albo Mistral. Zobaczysz jednak, że do takiego zadania nasza prosta sieć jest całkiem wystarczająca.

In [20]:
history = model.fit(x_train, y_train, epochs=10, batch_size=128, validation_data=(x_test, y_test))

Epoch 1/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.5859 - loss: 0.7317 - val_accuracy: 0.7084 - val_loss: 0.6454
Epoch 2/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.7625 - loss: 0.5908 - val_accuracy: 0.7923 - val_loss: 0.5499
Epoch 3/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.8109 - loss: 0.5283 - val_accuracy: 0.8212 - val_loss: 0.5091
Epoch 4/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.8286 - loss: 0.4960 - val_accuracy: 0.8364 - val_loss: 0.4834
Epoch 5/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.8412 - loss: 0.4739 - val_accuracy: 0.8428 - val_loss: 0.4655
Epoch 6/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.8501 - loss: 0.4570 - val_accuracy: 0.8364 - val_loss: 0.4621
Epoch 7/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.8529 - loss: 0.4454 - val_accuracy: 0.8314 - val_loss: 0.4639
Epoch 8/10
196/196 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.8582 - loss: 0.4349 - val_accu

In [21]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 16)        │        80,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3200)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         3,201 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 249,605 (975.02 KB)

 Trainable params: 83,201 (325.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 166,404 (650.02 KB)

Zbudowany model składa się z trzech warstw: Embedding, Flatten oraz Dense. Warstwa Embedding przekształca indeksy słów na 16-wymiarowe wektory reprezentujące znaczenie słów. Następnie warstwa Flatten zamienia macierz embeddingów na jednowymiarowy wektor, który jest przekazywany do pojedynczego neuronu z funkcją aktywacji sigmoid, odpowiedzialnego za klasyfikację binarną recenzji jako pozytywnej lub negatywnej. Model posiada 83 201 trenowalnych parametrów, z czego 80 000 należy do warstwy Embedding, a 3 201 do warstwy wyjściowej Dense. Dzięki niewielkiej liczbie parametrów model jest stosunkowo prosty i może być trenowany szybko nawet na procesorze CPU.

Za pomocą biblioteki sklearn możemy uzyskać dokładniejszy raport skuteczności modelu. Zobaczymy, że model nie ma specjalnych preferencji. Przy ocenie zarówno pozytywnych, jak i negatywnych komentarzy, ma rację w około 85% przypadków.

In [23]:
from sklearn.metrics import classification_report  # import funkcji do generowania raportu klasyfikacji

# Przewidywanie klas dla zbioru testowego
y_pred = (model.predict(x_test) > 0.5).astype("int32")  # zamiana prawdopodobieństw na klasy 0 lub 1 przy progu 0.5

# Wyświetlenie raportu klasyfikacji
print(
    classification_report(
        y_test,  # rzeczywiste etykiety
        y_pred,  # etykiety przewidziane przez model
        target_names=['Negative', 'Positive']  # nazwy klas
    )
)

782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
              precision    recall  f1-score   support

    Negative       0.86      0.86      0.86     12500
    Positive       0.86      0.86      0.86     12500

    accuracy                           0.86     25000
   macro avg       0.86      0.86      0.86     25000
weighted avg       0.86      0.86      0.86     25000



Model sieci neuronowej osiągnął 86% dokładności (accuracy) na zbiorze testowym. Dla obu klas – recenzji pozytywnych i negatywnych – uzyskano identyczne wartości Precision, Recall oraz F1-score równe 0.86, co świadczy o zrównoważonym działaniu klasyfikatora.

Wartość Precision na poziomie 0.86 oznacza, że 86% recenzji przypisanych do danej klasy zostało sklasyfikowanych poprawnie. Recall równy 0.86 wskazuje, że model poprawnie wykrył 86% wszystkich recenzji należących do każdej z klas. F1-score wynoszący 0.86 potwierdza dobrą równowagę pomiędzy precyzją i czułością modelu.

Ponieważ zbiór danych zawiera jednakową liczbę recenzji pozytywnych i negatywnych, wartości macro avg oraz weighted avg są identyczne. Uzyskane wyniki świadczą o tym, że model skutecznie klasyfikuje sentyment recenzji i nie wykazuje wyraźnego faworyzowania żadnej z klas.

# Predykcja

Przetestujmy zatem, jak radzi sobie nasz wytrenowany model na komentarzach, które podamy mu sami! Poniższa metoda encode_text zakoduje dla nas tekst do wektora.

In [24]:
# Funkcja przygotowująca własny tekst do klasyfikacji przez model
def encode_text(text):

    import re  # import biblioteki do wyrażeń regularnych

    # Funkcja zamieniająca pojedyncze słowo na jego indeks ze słownika IMDB
    def get_word_index(word):

        w_idx = index.get(word, -1)  # pobranie indeksu słowa lub -1, jeśli słowo nie istnieje

        return w_idx + 3 if w_idx <= num_words else 2  # przesunięcie indeksu o 3 lub zwrócenie indeksu słowa nieznanego (<UNK>)

    index = imdb.get_word_index()  # pobranie słownika IMDB (słowo -> indeks)

    text = re.sub(
        r'[^a-z ]',
        '',
        text.lower()
    )  # zamiana liter na małe oraz usunięcie znaków specjalnych i cyfr

    encoded = [
        get_word_index(word)
        for word in text.split(" ")
    ]  # zamiana każdego słowa na odpowiadający mu indeks

    return pad_sequences(
        [[1] + list(encoded)],
        maxlen=maxlen
    )  # dodanie znacznika początku (<START>) i dopasowanie długości sekwencji do maxlen

Funkcja encode_text() przygotowuje własny tekst do klasyfikacji przez wytrenowaną sieć neuronową. Wykonuje czyszczenie tekstu, zamienia słowa na odpowiadające im indeksy ze słownika IMDB, dodaje znacznik początku recenzji (<START>) oraz dopasowuje długość sekwencji do wymaganej przez model za pomocą pad_sequences()

Co robi funkcja krok po kroku?
- Pobiera słownik IMDB (word → index).
- Zamienia wszystkie litery na małe.
- Usuwa znaki specjalne, cyfry i interpunkcję.
- Zamienia każde słowo na odpowiadający mu numer.
- Dodaje na początku indeks 1, który oznacza znacznik <START>.
- Dopasowuje długość tekstu do 200 słów poprzez skróc

Teraz pozostaje już tylko wykonać metodę predict. Spróbuj wpisać własne komentarze i zobaczyć, jak się sprawuje. Pamiętaj, model zadziała jedynie dla języka angielskiego!

In [25]:
# Zakodowanie własnego tekstu do postaci wymaganej przez model
encoded_text = encode_text("last winter I was walking a lot")  # zamiana tekstu na sekwencję indeksów oraz dopasowanie długości do maxlen

print(encoded_text)  # wyświetlenie zakodowanej reprezentacji tekstu

# Przewidywanie sentymentu przez wytrenowany model
prediction = model.predict(encoded_text)  # zwrócenie prawdopodobieństwa klasy Positive

# Wyświetlenie wyniku klasyfikacji
print(
    f'Comment is {"Positive" if prediction > 0.5 else "Negative"}. Score: {prediction}'
)  # przypisanie klasy na podstawie progu 0.5 oraz wyświetlenie uzyskanego prawdopodobieństwa

[[   0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    1  236 34

Kod wykorzystuje wcześniej zdefiniowaną funkcję encode_text() do przygotowania własnego tekstu do klasyfikacji. Następnie wytrenowany model przewiduje sentyment recenzji, zwracając prawdopodobieństwo przynależności do klasy Positive. Jeżeli wartość jest większa niż 0.5, recenzja zostaje uznana za pozytywną, w przeciwnym razie za negatywną.

Dlaczego próg wynosi 0.5?

Warstwa wyjściowa modelu wykorzystuje funkcję aktywacji sigmoid, która zwraca wartość z przedziału 0–1:

wartość bliska 0 → recenzja negatywna,
wartość bliska 1 → recenzja pozytywna.

Najczęściej stosowanym progiem decyzyjnym jest 0.5:

prediction > 0.5 → Positive,
prediction ≤ 0.5 → Negative.

Dzięki temu model zamienia zwrócone prawdopodobieństwo na konkretną klasę.